Per-Ticker HMM + TFT Training

Each ticker gets its own fitted HMM and three TFT models (1d/3d/5d).

ETFs missing the 2008 crisis (start > 2009) use the frozen ^NSEI HMM
so their regime labels are consistent with what the TFT was trained on.
Stocks fit their own HMM from scratch.

Saved to `models/{TICKER}/`: hmm_model.pkl, state_mapping.json, tft_target_*.pt

In [1]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

import sys
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from hmmlearn.hmm import GaussianHMM
import lightning.pytorch as pl
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.data.encoders import NaNLabelEncoder
from pytorch_forecasting.metrics import MAE

from src.labeling import add_returns
from src.volatility import add_close_to_close_volatility
from src.features import add_lagged_returns, add_moving_average_features

warnings.filterwarnings("ignore")
import yfinance as yf

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

with open(PROJECT_ROOT / "tickers.json") as f:
    TICKERS = json.load(f)
print(f"Tickers to train: {len(TICKERS)}")

# ETFs that start after the 2008 crisis and lack enough High-regime history
# for their own HMM to converge meaningfully. These will borrow the
# frozen ^NSEI HMM labels instead.
NSEI_HMM_TICKERS = {"NIFTYBEES.NS", "GOLDBEES.NS"}


Device: cuda
Tickers to train: 15


## Helper functions

In [2]:
def fetch_and_prepare(ticker: str) -> pd.DataFrame:
    df = yf.download(ticker, start="2005-01-01", auto_adjust=True, progress=False)
    if df.empty:
        raise ValueError(f"No data for {ticker}")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
    df = df.sort_values("Date").dropna().reset_index(drop=True)
    df = add_returns(df)
    df = add_close_to_close_volatility(df, window=20)
    df = add_lagged_returns(df, lags=(1, 5, 10))
    df = add_moving_average_features(df, windows=(5, 10, 20))
    df["vol_cc_lag_1"] = df["vol_cc"].shift(1)
    df = df.dropna().reset_index(drop=True)
    return df


In [3]:
def fit_hmm_for_ticker(df: pd.DataFrame, save_dir: Path):
    hmm_input = df[["return", "vol_cc"]].values
    model = GaussianHMM(
        n_components=3,
        covariance_type="full",
        n_iter=1000,
        random_state=42,
    )
    model.fit(hmm_input)
    states = model.predict(hmm_input)
    df = df.copy()
    df["state"] = states

    state_vol = df.groupby("state")["vol_cc"].mean().sort_values()
    state_mapping = {
        int(state_vol.index[0]): "Low",
        int(state_vol.index[1]): "Medium",
        int(state_vol.index[2]): "High",
    }
    df["hmm_regime"] = df["state"].map(state_mapping)

    probs = model.predict_proba(hmm_input)
    df["P_state0"] = probs[:, 0]
    df["P_state1"] = probs[:, 1]
    df["P_state2"] = probs[:, 2]
    df["state_entropy"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)

    save_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, save_dir / "hmm_model.pkl")
    with open(save_dir / "state_mapping.json", "w") as f:
        json.dump(state_mapping, f)

    counts = df["hmm_regime"].value_counts().to_dict()
    print(f"  HMM: {counts}")
    return df, model, state_mapping


def apply_nsei_hmm(df: pd.DataFrame, save_dir: Path) -> pd.DataFrame:
    """
    For ETFs missing 2008 crisis data, we load the frozen ^NSEI HMM
    and apply it to this ticker's [return, vol_cc] features.
    Saves the same artifacts so the pipeline treats it identically.
    """
    nsei_dir = PROJECT_ROOT / "models" / "NSEI"
    if not (nsei_dir / "hmm_model.pkl").exists():
        raise FileNotFoundError(
            "models/NSEI/hmm_model.pkl not found. "
            "Run the NSEI training block first before other ETFs."
        )
    model = joblib.load(nsei_dir / "hmm_model.pkl")
    with open(nsei_dir / "state_mapping.json") as f:
        state_mapping = json.load(f)
    # state_mapping keys are strings in JSON, need int
    state_mapping = {int(k): v for k, v in state_mapping.items()}

    hmm_input = df[["return", "vol_cc"]].values
    states = model.predict(hmm_input)
    df = df.copy()
    df["state"] = states
    df["hmm_regime"] = df["state"].map(state_mapping)

    probs = model.predict_proba(hmm_input)
    df["P_state0"] = probs[:, 0]
    df["P_state1"] = probs[:, 1]
    df["P_state2"] = probs[:, 2]
    df["state_entropy"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)

    # save copies into this ticker's folder so pipeline loads them normally
    save_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, save_dir / "hmm_model.pkl")
    with open(save_dir / "state_mapping.json", "w") as f:
        json.dump({str(k): v for k, v in state_mapping.items()}, f)

    counts = df["hmm_regime"].value_counts().to_dict()
    print(f"  HMM (from ^NSEI): {counts}")
    return df, model, state_mapping


In [4]:
def build_transition_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    is_high = (df["hmm_regime"] == "High").astype(int)
    df["target_1d"] = is_high.shift(-1)
    df["target_3d"] = pd.concat(
        [is_high.shift(-i) for i in range(1, 4)], axis=1
    ).max(axis=1)
    df["target_5d"] = pd.concat(
        [is_high.shift(-i) for i in range(1, 6)], axis=1
    ).max(axis=1)
    for col in ["target_1d", "target_3d", "target_5d"]:
        df[col] = df[col].astype("Int64")
    return df

UNKNOWN_REALS = [
    "return", "vol_cc", "vol_cc_lag_1",
    "ret_lag_1", "ret_lag_5", "ret_lag_10",
    "ma_ratio_5", "ma_ratio_10", "ma_ratio_20",
]

def make_tft_datasets(df: pd.DataFrame, ticker: str):
    df = df.copy()
    df["time_idx"] = df.index.astype(int)
    df["series"] = ticker
    for col in ["target_1d", "target_3d", "target_5d"]:
        df[col] = df[col].astype(int)

    split = int(len(df) * 0.80)
    train_df = df.iloc[:split].copy()
    val_df = df.iloc[split:].copy()

    datasets = {}
    for horizon in ["target_1d", "target_3d", "target_5d"]:
        training = TimeSeriesDataSet(
            train_df,
            time_idx="time_idx",
            target=horizon,
            group_ids=["series"],
            max_encoder_length=30,
            max_prediction_length=1,
            time_varying_unknown_reals=UNKNOWN_REALS,
            time_varying_known_reals=["time_idx"],
            time_varying_known_categoricals=["hmm_regime"],
            target_normalizer=None,
            categorical_encoders={"hmm_regime": NaNLabelEncoder(add_nan=False)},
        )
        validation = TimeSeriesDataSet.from_dataset(
            training, val_df, stop_randomization=True
        )
        datasets[horizon] = (training, validation)
        print(f"  {horizon}: {len(training)} train, {len(validation)} val")
    return datasets


In [5]:
def train_tft_for_horizon(training_ds, val_ds, target_col, df, max_epochs=20):
    pos = df[target_col].mean()
    pos_weight = torch.tensor((1 - pos) / (pos + 1e-9))
    print(f"  pos_weight ({target_col}): {pos_weight:.2f}")

    tft = TemporalFusionTransformer.from_dataset(
        training_ds,
        learning_rate=0.01,
        hidden_size=16,
        attention_head_size=1,
        dropout=0.1,
        hidden_continuous_size=8,
        output_size=1,
        loss=MAE(),
        log_interval=-1,
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    optimizer = Adam(tft.parameters(), lr=0.01)
    scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    train_loader = training_ds.to_dataloader(train=True, batch_size=64, num_workers=0)
    val_loader = val_ds.to_dataloader(train=False, batch_size=64, num_workers=0)

    best_val, best_state = float("inf"), None

    for epoch in range(max_epochs):
        tft.train()
        train_loss = 0.0
        for x, (y, _) in train_loader:
            x = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in x.items()}
            y = y.to(DEVICE).float()
            optimizer.zero_grad()
            loss = criterion(tft(x)["prediction"].squeeze(-1), y)
            loss.backward()
            nn.utils.clip_grad_norm_(tft.parameters(), 0.1)
            optimizer.step()
            train_loss += loss.item()

        tft.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, (y, _) in val_loader:
                x = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in x.items()}
                y = y.to(DEVICE).float()
                val_loss += criterion(tft(x)["prediction"].squeeze(-1), y).item()

        scheduler.step(val_loss)
        if (epoch + 1) % 5 == 0:
            print(f"    epoch {epoch+1:02d} | train {train_loss/len(train_loader):.4f} | val {val_loss/len(val_loader):.4f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in tft.state_dict().items()}

    tft.load_state_dict(best_state)
    tft = tft.to(DEVICE)
    print(f"  best val loss: {best_val/len(val_loader):.4f}")
    return tft


## Training loop

In [13]:
MODELS_DIR = PROJECT_ROOT / "models"
summary = {}

# sort so ^NSEI-dependent ETFs train after NSEI itself
# we handle this by putting NIFTYBEES/GOLDBEES last
nsei_etfs = [t for t in TICKERS if t["ticker"] in NSEI_HMM_TICKERS]
others = [t for t in TICKERS if t["ticker"] not in NSEI_HMM_TICKERS]

# first train ^NSEI standalone so its model is available
print("\nPre-training: ^NSEI (shared HMM source for ETFs)")
nsei_save = MODELS_DIR / "NSEI"
try:
    nsei_df = fetch_and_prepare("^NSEI")
    print(f"  Data: {len(nsei_df)} rows, {nsei_df['Date'].min().date()} to {nsei_df['Date'].max().date()}")
    nsei_df, _, _ = fit_hmm_for_ticker(nsei_df, nsei_save)
    print("  ^NSEI HMM saved.")
except Exception as e:
    print(f"  FAILED ^NSEI: {e} -- ETFs will fail too")

# now train all tickers: stocks first, ETFs last
for t in others + nsei_etfs:
    ticker = t["ticker"]
    clean = ticker.replace(".", "_")
    save_dir = MODELS_DIR / clean

    print(f"\n{'='*50}")
    print(f"Training: {ticker} ({t['name']})")
    print(f"{'='*50}")

    try:
        df = fetch_and_prepare(ticker)
        print(f"  Data: {len(df)} rows, {df['Date'].min().date()} to {df['Date'].max().date()}")

        if ticker in NSEI_HMM_TICKERS:
            df, hmm_model, state_mapping = apply_nsei_hmm(df, save_dir)
        else:
            df, hmm_model, state_mapping = fit_hmm_for_ticker(df, save_dir)

        df = build_transition_targets(df)
        df = df.dropna(subset=["target_1d", "target_3d", "target_5d"]).reset_index(drop=True)
        print(f"  Target balance: 1d={df['target_1d'].mean():.1%}  3d={df['target_3d'].mean():.1%}  5d={df['target_5d'].mean():.1%}")

        datasets = make_tft_datasets(df, ticker)

        for horizon, (tr_ds, va_ds) in datasets.items():
            print(f"  Training {horizon}...")
            model = train_tft_for_horizon(tr_ds, va_ds, horizon, df)
            torch.save(model.state_dict(), save_dir / f"tft_{horizon}.pt")
            print(f"  Saved tft_{horizon}.pt")

        summary[ticker] = "OK"
        print(f"  {ticker} complete.")

    except Exception as e:
        summary[ticker] = f"FAILED: {e}"
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()

print("\nSummary:")
for ticker, status in summary.items():
    print(f"  {ticker:<25} {status}")



Pre-training: ^NSEI (shared HMM source for ETFs)
  Data: 4518 rows, 2007-10-17 to 2026-03-18
  HMM: {'Medium': 1678, 'Low': 1672, 'High': 1168}
  ^NSEI HMM saved.

Training: HDFCBANK.NS (HDFC Bank)
  Data: 5215 rows, 2005-02-03 to 2026-03-18
  HMM: {'Low': 2810, 'Medium': 1794, 'High': 611}
  Target balance: 1d=11.7%  3d=12.2%  5d=12.7%
  target_1d: 4141 train, 1013 val
  target_3d: 4141 train, 1013 val
  target_5d: 4141 train, 1013 val
  Training target_1d...
  pos_weight (target_1d): 7.55
    epoch 05 | train 0.7113 | val 0.1062
    epoch 10 | train 0.1210 | val 0.1035
    epoch 15 | train 0.1129 | val 0.1055
    epoch 20 | train 0.1193 | val 0.1055
  best val loss: 0.0967
  Saved tft_target_1d.pt
  Training target_3d...
  pos_weight (target_3d): 7.20
    epoch 05 | train 0.2405 | val 0.2269
    epoch 10 | train 0.2340 | val 0.2347
    epoch 15 | train 0.2031 | val 0.2486
    epoch 20 | train 0.1878 | val 0.2072
  best val loss: 0.2071
  Saved tft_target_3d.pt
  Training target_5d..

Model is not converging.  Current: 19829.989108588994 is not greater than 19829.990668015496. Delta is -0.0015594265023537446


  HMM: {'Low': 2317, 'Medium': 2165, 'High': 733}
  Target balance: 1d=14.1%  3d=14.8%  5d=15.5%
  target_1d: 4141 train, 1013 val
  target_3d: 4141 train, 1013 val
  target_5d: 4141 train, 1013 val
  Training target_1d...
  pos_weight (target_1d): 6.11
    epoch 05 | train 0.1698 | val 0.0069
    epoch 10 | train 0.1581 | val 0.0040
    epoch 15 | train 0.1232 | val 0.0032
    epoch 20 | train 0.0941 | val 0.0048
  best val loss: 0.0021
  Saved tft_target_1d.pt
  Training target_3d...
  pos_weight (target_3d): 5.76
    epoch 05 | train 0.2784 | val 0.0361
    epoch 10 | train 0.2642 | val 0.0853
    epoch 15 | train 0.2321 | val 0.0403
    epoch 20 | train 0.2004 | val 0.0489
  best val loss: 0.0290
  Saved tft_target_3d.pt
  Training target_5d...
  pos_weight (target_5d): 5.44
    epoch 05 | train 0.4023 | val 0.0550
    epoch 10 | train 0.3730 | val 0.0958
    epoch 15 | train 0.3004 | val 0.4978
    epoch 20 | train 0.2203 | val 0.4979
  best val loss: 0.0317
  Saved tft_target_5d.

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found
Traceback (most recent call last):
  File "C:\Users\Aryavart\AppData\Local\Temp\ipykernel_28440\3906517838.py", line 31, in <module>
    df = fetch_and_prepare(ticker)
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Aryavart\AppData\Local\Temp\ipykernel_28440\3342736018.py", line 4, in fetch_and_prepare
    raise ValueError(f"No data for {ticker}")
ValueError: No data for TATAMOTORS.NS


  ERROR: No data for TATAMOTORS.NS

Training: TATASTEEL.NS (Tata Steel)
  Data: 5215 rows, 2005-02-03 to 2026-03-18
  HMM: {'Medium': 1796, 'Low': 1792, 'High': 1627}
  Target balance: 1d=31.2%  3d=32.5%  5d=33.8%
  target_1d: 4141 train, 1013 val
  target_3d: 4141 train, 1013 val
  target_5d: 4141 train, 1013 val
  Training target_1d...
  pos_weight (target_1d): 2.20
    epoch 05 | train 0.1077 | val 0.0790
    epoch 10 | train 0.1005 | val 0.0819
    epoch 15 | train 0.0964 | val 0.0735
    epoch 20 | train 0.0908 | val 0.0805
  best val loss: 0.0734
  Saved tft_target_1d.pt
  Training target_3d...
  pos_weight (target_3d): 2.08
    epoch 05 | train 0.2018 | val 0.1526
    epoch 10 | train 0.1833 | val 0.1480
    epoch 15 | train 0.1780 | val 0.1460
    epoch 20 | train 0.1593 | val 0.1481
  best val loss: 0.1456
  Saved tft_target_3d.pt
  Training target_5d...
  pos_weight (target_5d): 1.96
    epoch 05 | train 0.2572 | val 0.2304
    epoch 10 | train 0.2321 | val 0.2051
    epoch 1

In [6]:
# TATAMOTORS.NS returned 404 from yfinance
# Replacing with MARUTI.NS (Maruti Suzuki)

MODELS_DIR = PROJECT_ROOT / "models"
summary = {}

ticker = "MARUTI.NS"
clean = ticker.replace(".", "_")
save_dir = MODELS_DIR / clean

print(f"Training replacement ticker: {ticker}")

df = fetch_and_prepare(ticker)
print(f"  Data: {len(df)} rows, {df['Date'].min().date()} to {df['Date'].max().date()}")

df, hmm_model, state_mapping = fit_hmm_for_ticker(df, save_dir)

df = build_transition_targets(df)
df = df.dropna(subset=["target_1d", "target_3d", "target_5d"]).reset_index(drop=True)
print(f"  Target balance: 1d={df['target_1d'].mean():.1%}  3d={df['target_3d'].mean():.1%}  5d={df['target_5d'].mean():.1%}")

datasets = make_tft_datasets(df, ticker)

for horizon, (tr_ds, va_ds) in datasets.items():
    print(f"  Training {horizon}...")
    model = train_tft_for_horizon(tr_ds, va_ds, horizon, df)
    torch.save(model.state_dict(), save_dir / f"tft_{horizon}.pt")
    print(f"  Saved.")

print("MARUTI.NS complete.")

Training replacement ticker: MARUTI.NS
  Data: 5217 rows, 2005-02-03 to 2026-03-20


Model is not converging.  Current: 21733.021459059426 is not greater than 21733.026898273845. Delta is -0.005439214419311611


  HMM: {'Low': 2377, 'Medium': 1962, 'High': 878}
  Target balance: 1d=16.8%  3d=17.8%  5d=18.7%
  target_1d: 4142 train, 1014 val
  target_3d: 4142 train, 1014 val
  target_5d: 4142 train, 1014 val
  Training target_1d...
  pos_weight (target_1d): 4.95
    epoch 05 | train 0.1777 | val 0.0090
    epoch 10 | train 0.1581 | val 0.0094
    epoch 15 | train 0.1520 | val 0.0101
    epoch 20 | train 0.1495 | val 0.0091
  best val loss: 0.0079
  Saved.
  Training target_3d...
  pos_weight (target_3d): 4.63
    epoch 05 | train 0.2773 | val 0.0273
    epoch 10 | train 0.2686 | val 0.0371
    epoch 15 | train 0.2569 | val 0.0291
    epoch 20 | train 0.2528 | val 0.0276
  best val loss: 0.0273
  Saved.
  Training target_5d...
  pos_weight (target_5d): 4.34
    epoch 05 | train 0.3297 | val 0.0386
    epoch 10 | train 0.3201 | val 0.0594
    epoch 15 | train 0.2750 | val 0.0247
    epoch 20 | train 0.2454 | val 0.0260
  best val loss: 0.0205
  Saved.
MARUTI.NS complete.


In [7]:
# Replacing TCS.NS and RELIANCE.NS with more volatile sector counterparts
# TCS -> HCLTECH.NS (IT)
# RELIANCE -> BPCL.NS (Energy/Refining)

REPLACEMENTS = [
    {"ticker": "HCLTECH.NS", "name": "HCL Technologies"},
    {"ticker": "BPCL.NS", "name": "Bharat Petroleum"}
]

for item in REPLACEMENTS:
    ticker = item["ticker"]
    clean = ticker.replace(".", "_")
    save_dir = MODELS_DIR / clean
    
    print(f"\n{'='*50}")
    print(f"Training Replacement: {ticker} ({item['name']})")
    print(f"{'='*50}")

    try:
        # 1. Prepare Data
        df = fetch_and_prepare(ticker)
        print(f"  Data: {len(df)} rows")

        # 2. Fit HMM (Standard fit, not shared NSEI)
        df, hmm_model, state_mapping = fit_hmm_for_ticker(df, save_dir)

        # 3. Build Targets
        df = build_transition_targets(df)
        df = df.dropna(subset=["target_1d", "target_3d", "target_5d"]).reset_index(drop=True)
        print(f"  Target balance: 1d={df['target_1d'].mean():.1%}  3d={df['target_3d'].mean():.1%}")

        # 4. Create Datasets
        datasets = make_tft_datasets(df, ticker)

        # 5. Train TFT for each horizon
        for horizon, (tr_ds, va_ds) in datasets.items():
            print(f"  Training {horizon}...")
            model = train_tft_for_horizon(tr_ds, va_ds, horizon, df, max_epochs=20)
            torch.save(model.state_dict(), save_dir / f"tft_{horizon}.pt")
            print(f"  Saved {horizon} to {save_dir}")

        print(f"Successfully replaced and trained {ticker}.")

    except Exception as e:
        print(f"  ERROR training {ticker}: {e}")


Training Replacement: HCLTECH.NS (HCL Technologies)
  Data: 5218 rows
  HMM: {'Low': 2048, 'Medium': 1910, 'High': 1260}
  Target balance: 1d=24.2%  3d=25.2%
  target_1d: 4143 train, 1014 val
  target_3d: 4143 train, 1014 val
  target_5d: 4143 train, 1014 val
  Training target_1d...
  pos_weight (target_1d): 3.14
    epoch 05 | train 0.1233 | val 0.0511
    epoch 10 | train 0.1166 | val 0.0509
    epoch 15 | train 0.1138 | val 0.0503
    epoch 20 | train 0.1043 | val 0.0485
  best val loss: 0.0485
  Saved target_1d to C:\Users\Aryavart\stock-regime-ml\models\HCLTECH_NS
  Training target_3d...
  pos_weight (target_3d): 2.96
    epoch 05 | train 0.2138 | val 0.1038
    epoch 10 | train 0.2080 | val 0.1029
    epoch 15 | train 0.1892 | val 0.1053
    epoch 20 | train 0.1631 | val 0.1040
  best val loss: 0.1000
  Saved target_3d to C:\Users\Aryavart\stock-regime-ml\models\HCLTECH_NS
  Training target_5d...
  pos_weight (target_5d): 2.80
    epoch 05 | train 0.2674 | val 0.1486
    epoch 1

## Saved artifacts

models/{TICKER}/: hmm_model.pkl, state_mapping.json, tft_target_1d.pt, tft_target_3d.pt, tft_target_5d.pt

ETFs in NSEI_HMM_TICKERS have the ^NSEI model copied into their folder —
the pipeline loads them identically to stock models.

We will retrain quarterly or after major market structural events.